# 08 — Model improvements: historical validation (Good → Fair or Poor)

This notebook compares a five-predictor Cox PH model with a seven-predictor Cox PH model adding reconstruction features. Validation follows bridges Good at the end of 2016 through 2017–2020. Both Fair and Poor are deterioration events. The final 2021–2025 test set is **not** used for model selection.

**Important:** The historical validation C-index below is an exploratory, censoring-aware *post-entry hazard ranking*. It does not fully handle different delayed entry durations and is not a direct measure of accuracy in predicted remaining years. The final standardised comparison belongs in Notebook 11. This notebook was checked structurally but not executed on the local NBI data.

# Survival Model Improvements

## 1. Feature Investigation

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Define project directories.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent
FINAL = PROJECT / "data" / "final"
RESULTS = PROJECT / "results"

# Load the prepared training and testing datasets.
train_data = pd.read_parquet(
    FINAL / "bridge_survival_train.parquet"
)

test_data = pd.read_parquet(
    FINAL / "bridge_survival_test_aligned.parquet"
)  # For feature inspection only; never used for model selection

# Identify columns available in both datasets.
common_columns = sorted(
    set(train_data.columns) & set(test_data.columns)
)

print("Training bridges:", f"{len(train_data):,}")
print("Testing bridges:", f"{len(test_data):,}")

print("\nTraining columns:")
print(train_data.columns.tolist())

print("\nColumns available in both datasets:")
print(common_columns)


Training bridges: 498,875
Testing bridges: 171,766

Training columns:
['bridge_id', 'start_year', 'bridge_age', 'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B', 'STRUCTURE_LEN_MT_049', 'end_year', 'event', 'stop_reason', 'last_year', 'duration']

Columns available in both datasets:
['ADT_029', 'STRUCTURE_KIND_043A', 'STRUCTURE_LEN_MT_049', 'STRUCTURE_TYPE_043B', 'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'bridge_age', 'bridge_id', 'duration', 'end_year', 'event', 'start_year']


In [2]:

# Examine reconstruction years in the training and testing datasets.
column = "YEAR_RECONSTRUCTED_106"

print("Training reconstruction years:")
print(train_data[column].describe())
print("\nMost common training values:")
print(train_data[column].value_counts(dropna=False).head(15))

print("\nTesting reconstruction years:")
print(test_data[column].describe())
print("\nMost common testing values:")
print(test_data[column].value_counts(dropna=False).head(15))

# Check whether reconstruction dates occur after the observation start.
for name, data in [("Training", train_data), ("Testing", test_data)]:
    reconstruction_year = pd.to_numeric(data[column], errors="coerce")
    start_year = pd.to_numeric(data["start_year"], errors="coerce")

    future_reconstructions = (
        reconstruction_year.notna()
        & (reconstruction_year > start_year)
    )

    print(
        f"\n{name} reconstruction dates after start year:",
        f"{future_reconstructions.sum():,}"
    )


Training reconstruction years:
count    473569.000000
mean        179.324586
std         569.199295
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        8700.000000
Name: YEAR_RECONSTRUCTED_106, dtype: float64

Most common training values:
YEAR_RECONSTRUCTED_106
0.0       430632
NaN        25306
1988.0      2252
1989.0      2210
1987.0      2109
1986.0      2066
1990.0      1975
1985.0      1784
1984.0      1715
1991.0      1424
1983.0      1381
1980.0      1257
1982.0      1183
1981.0      1174
1979.0       935
Name: count, dtype: int64

Testing reconstruction years:
count    138561.000000
mean        143.120784
std         514.629911
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        2029.000000
Name: YEAR_RECONSTRUCTED_106, dtype: float64

Most common testing values:
YEAR_RECONSTRUCTED_106
0.0       128613
NaN        33205
1990.0       319
1989.0       274
2008.0       269
2009.0   

In [3]:

# Check reconstruction year validity in both datasets.
for name, data in [("Training", train_data), ("Testing", test_data)]:
    reconstruction = pd.to_numeric(
        data["YEAR_RECONSTRUCTED_106"], errors="coerce"
    )
    built = pd.to_numeric(
        data["YEAR_BUILT_027"], errors="coerce"
    )
    start = pd.to_numeric(
        data["start_year"], errors="coerce"
    )

    valid_reconstruction = (
        reconstruction.between(1800, 2025)
        & (reconstruction >= built)
        & (reconstruction <= start)
    )

    print(f"\n{name}:")
    print("Valid reconstruction records:", f"{valid_reconstruction.sum():,}")
    print("Reconstruction year = 0:", f"{(reconstruction == 0).sum():,}")
    print("Missing reconstruction year:", f"{reconstruction.isna().sum():,}")
    print(
        "Invalid positive reconstruction years:",
        f"{((reconstruction > 0) & ~valid_reconstruction).sum():,}"
    )
    print(
        "Percentage with valid reconstruction:",
        f"{valid_reconstruction.mean() * 100:.2f}%"
    )



Training:
Valid reconstruction records: 42,072
Reconstruction year = 0: 430,632
Missing reconstruction year: 25,306
Invalid positive reconstruction years: 865
Percentage with valid reconstruction: 8.43%

Testing:
Valid reconstruction records: 6,875
Reconstruction year = 0: 128,613
Missing reconstruction year: 33,205
Invalid positive reconstruction years: 3,073
Percentage with valid reconstruction: 4.00%


In [4]:

# Examine reconstruction timing relative to the observation start.
for name, data in [("Training", train_data), ("Testing", test_data)]:
    reconstruction = pd.to_numeric(
        data["YEAR_RECONSTRUCTED_106"], errors="coerce"
    )
    built = pd.to_numeric(
        data["YEAR_BUILT_027"], errors="coerce"
    )
    start = pd.to_numeric(
        data["start_year"], errors="coerce"
    )

    valid = (
        reconstruction.between(1800, 2025)
        & (reconstruction >= built)
        & (reconstruction <= start)
    )

    years_since = start[valid] - reconstruction[valid]

    print(f"\n{name}:")
    print("Observation start years:")
    print(start.value_counts().sort_index().to_string())

    print("\nYears since reconstruction:")
    print(years_since.describe().round(2).to_string())

    print("\nReconstruction in observation start year:",
          int((years_since == 0).sum()))



Training:
Observation start years:
start_year
1992    253945
1993     27187
1994      6186
1995      9630
1996      5603
1997     14631
1998      5096
1999      5694
2000     12426
2001      7799
2002     15991
2003      5918
2004      6012
2005      5866
2006     16674
2007      5794
2008      5128
2009      5417
2010      5759
2011     15260
2012      5979
2013      9244
2014     19538
2015      4943
2016      8721
2017      5025
2018      4724
2019      4685

Years since reconstruction:
count    42072.00
mean        15.14
std         12.98
min          0.00
25%          5.00
50%         11.00
75%         22.00
max        129.00

Reconstruction in observation start year: 150

Testing:
Observation start years:
start_year
1992    41678
1993     5048
1994     1661
1995     2417
1996     1791
1997     4149
1998     1744
1999     2059
2000     3410
2001     2710
2002     6071
2003     2824
2004     3043
2005     3402
2006     5206
2007     3542
2008     3209
2009     3316
2010     3642
2

## 2. Reconstruction Feature Engineering

In [5]:

# Create reconstruction features using information available at observation start.
def add_reconstruction_features(data):
    data = data.copy()

    reconstruction = pd.to_numeric(
        data["YEAR_RECONSTRUCTED_106"], errors="coerce"
    )
    built = pd.to_numeric(
        data["YEAR_BUILT_027"], errors="coerce"
    )
    start = pd.to_numeric(
        data["start_year"], errors="coerce"
    )

    # Accept only plausible reconstruction dates known by observation start.
    valid = (
        reconstruction.between(1800, 2025)
        & (reconstruction >= built)
        & (reconstruction <= start)
    )

    # Distinguish recorded reconstructions from missing or invalid dates.
    data["was_reconstructed"] = valid.astype(int)
    data["years_since_reconstruction"] = (
        start - reconstruction
    ).where(valid, 0)

    return data

train_improved = add_reconstruction_features(train_data)
test_improved = add_reconstruction_features(test_data)

print("Training reconstructed:", train_improved["was_reconstructed"].sum())
print("Testing reconstructed:", test_improved["was_reconstructed"].sum())

print("\nNew features:")
print(train_improved[
    ["was_reconstructed", "years_since_reconstruction"]
].describe())


Training reconstructed: 42072
Testing reconstructed: 6875

New features:
       was_reconstructed  years_since_reconstruction
count      498875.000000               498875.000000
mean            0.084334                    1.276931
std             0.277888                    5.649532
min             0.000000                    0.000000
25%             0.000000                    0.000000
50%             0.000000                    0.000000
75%             0.000000                    0.000000
max             1.000000                  129.000000


In [6]:

# Validate the new reconstruction features.
new_features = [
    "was_reconstructed",
    "years_since_reconstruction"
]

for name, data in [
    ("Training", train_improved),
    ("Testing", test_improved)
]:
    print(f"\n{name} dataset:")

    print("Missing values:")
    print(data[new_features].isna().sum())

    print("Negative reconstruction ages:",
          (data["years_since_reconstruction"] < 0).sum())

    print("Invalid reconstruction indicators:",
          (~data["was_reconstructed"].isin([0, 1])).sum())

    print("Non-reconstructed bridges with positive reconstruction age:",
          (
              (data["was_reconstructed"] == 0)
              & (data["years_since_reconstruction"] > 0)
          ).sum())



Training dataset:
Missing values:
was_reconstructed             0
years_since_reconstruction    0
dtype: int64
Negative reconstruction ages: 0
Invalid reconstruction indicators: 0
Non-reconstructed bridges with positive reconstruction age: 0

Testing dataset:
Missing values:
was_reconstructed             0
years_since_reconstruction    0
dtype: int64
Negative reconstruction ages: 0
Invalid reconstruction indicators: 0
Non-reconstructed bridges with positive reconstruction age: 0


## 3. Improved Model Preparation

In [7]:

from sklearn.model_selection import train_test_split

# Original predictors used by the survival models.
numeric_predictors = [
    "bridge_age",
    "ADT_029",
    "STRUCTURE_LEN_MT_049"
]

categorical_predictors = [
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

# Additional reconstruction features.
reconstruction_predictors = [
    "was_reconstructed",
    "years_since_reconstruction"
]

all_predictors = (
    numeric_predictors
    + categorical_predictors
    + reconstruction_predictors
)

# Historical validation is constructed below using 2016 as cutoff.
# No random split is used for model selection.
print("Number of candidate predictors:", len(all_predictors))


Number of candidate predictors: 7


In [8]:
# The actual historical training and validation cohorts are built in Sections 4–6.
print("Using 2016 cutoff for historical validation, not a random split.")

Using 2016 cutoff for historical validation, not a random split.


In [9]:

# Examine whether a 2016 cutoff could support temporal validation.
cutoff_year = 2016

earlier_bridges = train_improved[
    train_improved["start_year"] <= cutoff_year
]

later_bridges = train_improved[
    train_improved["start_year"] > cutoff_year
]

print("Bridges starting by 2016:", f"{len(earlier_bridges):,}")
print("Bridges starting after 2016:", f"{len(later_bridges):,}")

print("\nEarlier bridges with outcomes extending beyond 2016:",
      f"{(earlier_bridges['end_year'] > cutoff_year).sum():,}")

print("\nLater bridges event counts:")
print(later_bridges["event"].value_counts())

print("\nLater bridges follow-up durations:")
print(later_bridges["duration"].describe().round(2))


Bridges starting by 2016: 484,441
Bridges starting after 2016: 14,434

Earlier bridges with outcomes extending beyond 2016: 187,045

Later bridges event counts:
event
0    14030
1      404
Name: count, dtype: int64

Later bridges follow-up durations:
count    14434.00
mean         2.00
std          0.82
min          1.00
25%          1.00
50%          2.00
75%          3.00
max          3.00
Name: duration, dtype: float64


In [10]:

# Locate available historical bridge data files.
data_directory = PROJECT / "data"

for path in sorted(data_directory.rglob("*")):
    if path.is_file() and path.suffix.lower() in [
        ".parquet", ".csv", ".pkl", ".pickle"
    ]:
        print(path.relative_to(PROJECT))


data/final/bridge_historical_validation.parquet
data/final/bridge_survival_dataset.parquet
data/final/bridge_survival_test.parquet
data/final/bridge_survival_test_aligned.parquet
data/final/bridge_survival_train.parquet
data/processed/nbi_1992_clean.parquet
data/processed/nbi_1993_clean.parquet
data/processed/nbi_1994_clean.parquet
data/processed/nbi_1995_clean.parquet
data/processed/nbi_1996_clean.parquet
data/processed/nbi_1997_clean.parquet
data/processed/nbi_1998_clean.parquet
data/processed/nbi_1999_clean.parquet
data/processed/nbi_2000_clean.parquet
data/processed/nbi_2001_clean.parquet
data/processed/nbi_2002_clean.parquet
data/processed/nbi_2003_clean.parquet
data/processed/nbi_2004_clean.parquet
data/processed/nbi_2005_clean.parquet
data/processed/nbi_2006_clean.parquet
data/processed/nbi_2007_clean.parquet
data/processed/nbi_2008_clean.parquet
data/processed/nbi_2009_clean.parquet
data/processed/nbi_2010_clean.parquet
data/processed/nbi_2011_clean.parquet
data/processed/nbi_2

In [11]:

import pyarrow.parquet as pq

# Inspect historical snapshots without loading the full datasets.
for year in [2016, 2017, 2020, 2021]:
    path = PROJECT / "data" / "processed" / f"nbi_{year}_clean.parquet"
    metadata = pq.read_metadata(path)

    print(f"\n{year} snapshot:")
    print("Rows:", f"{metadata.num_rows:,}")
    print("Columns:", metadata.num_columns)
    print("Column names:")
    print(metadata.schema.names)



2016 snapshot:
Rows: 614,379
Columns: 19
Column names:
['STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'YEAR_BUILT_027', 'ADT_029', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B', 'STRUCTURE_LEN_MT_049', 'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062', 'DATE_OF_INSPECT_090', 'YEAR_RECONSTRUCTED_106', 'bridge_id', 'condition_rating', 'condition', 'bridge_age', 'observation_year']

2017 snapshot:
Rows: 614,994
Columns: 19
Column names:
['STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'YEAR_BUILT_027', 'ADT_029', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B', 'STRUCTURE_LEN_MT_049', 'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062', 'DATE_OF_INSPECT_090', 'YEAR_RECONSTRUCTED_106', 'bridge_id', 'condition_rating', 'condition', 'bridge_age', 'observation_year']

2020 snapshot:
Rows: 618,454
Columns: 19
Column names:
['STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'YEAR_BUI

In [12]:

# Identify bridges in Good condition at the 2016 prediction cutoff.
snapshot_2016 = pd.read_parquet(
    PROJECT / "data" / "processed" / "nbi_2016_clean.parquet",
    columns=["bridge_id", "condition", "observation_year"]
)

good_2016 = snapshot_2016.loc[
    snapshot_2016["condition"] == "Good",
    "bridge_id"
].drop_duplicates()

print("Total bridges in 2016:", f"{len(snapshot_2016):,}")
print("Bridges in Good condition:", f"{len(good_2016):,}")
print("Unique Good-condition bridge IDs:", good_2016.nunique())
print("Percentage Good:",
      round(100 * len(good_2016) / len(snapshot_2016), 2))


Total bridges in 2016: 614,379
Bridges in Good condition: 291,414
Unique Good-condition bridge IDs: 291414
Percentage Good: 47.43


In [13]:

# Track the 2016 Good-condition bridges during the validation period.
validation_history = []

for year in range(2017, 2021):
    snapshot = pd.read_parquet(
        PROJECT / "data" / "processed" / f"nbi_{year}_clean.parquet",
        columns=["bridge_id", "condition", "observation_year"]
    )

    snapshot = snapshot[
        snapshot["bridge_id"].isin(good_2016)
    ]

    validation_history.append(snapshot)

    print(f"{year}: {len(snapshot):,} eligible bridge records")

validation_history = pd.concat(
    validation_history,
    ignore_index=True
)

print("\nTotal validation observations:",
      f"{len(validation_history):,}")

print("\nConditions observed:")
print(validation_history["condition"].value_counts(dropna=False))


2017: 290,275 eligible bridge records
2018: 289,789 eligible bridge records
2019: 289,198 eligible bridge records
2020: 288,739 eligible bridge records

Total validation observations: 1,158,001

Conditions observed:
condition
Good       1023070
Fair        131406
Poor          3522
Unknown          3
Name: count, dtype: int64


In [14]:

# Find each bridge's first observed condition other than Good.
history_sorted = validation_history.sort_values(
    ["bridge_id", "observation_year"]
)

first_non_good = (
    history_sorted[
        history_sorted["condition"] != "Good"
    ]
    .drop_duplicates(subset="bridge_id", keep="first")
)

print("Bridges with a non-Good observation:",
      f"{len(first_non_good):,}")

print("\nFirst non-Good condition:")
print(first_non_good["condition"].value_counts(dropna=False))

print("\nFirst non-Good observation year:")
print(
    pd.crosstab(
        first_non_good["observation_year"],
        first_non_good["condition"]
    )
)


Bridges with a non-Good observation: 54,071

First non-Good condition:
condition
Fair       52709
Poor        1359
Unknown        3
Name: count, dtype: int64

First non-Good observation year:
condition          Fair  Poor  Unknown
observation_year                      
2017              14313   355        2
2018              14263   382        1
2019              12297   304        0
2020              11836   318        0


In [15]:

# Check whether bridges were observed every year before deterioration.
first_deterioration = first_non_good[
    first_non_good["condition"].isin(["Fair", "Poor"])
][["bridge_id", "observation_year"]].copy()

first_deterioration = first_deterioration.rename(
    columns={"observation_year": "first_deterioration_year"}
)

# Count observations between 2017 and the first deterioration year.
observations_before_fair = (
    validation_history.merge(first_deterioration, on="bridge_id")
    .query("observation_year <= first_deterioration_year")
    .groupby("bridge_id")["observation_year"]
    .nunique()
)

# Expected observations include every year from 2017 to first deterioration.
expected_observations = (
    first_deterioration.set_index("bridge_id")["first_deterioration_year"] - 2016
)

missing_years = (
    expected_observations - observations_before_fair
)

print("Bridges first observed as Fair or Poor:", f"{len(first_deterioration):,}")
print("Bridges with complete annual observations:",
      f"{(missing_years == 0).sum():,}")
print("Bridges with observation gaps:",
      f"{(missing_years > 0).sum():,}")

print("\nNumber of missing years before first deterioration:")
print(missing_years.value_counts().sort_index())


Bridges first observed as Fair or Poor: 54,068
Bridges with complete annual observations: 54,046
Bridges with observation gaps: 22

Number of missing years before first deterioration:
0    54046
1       12
2        9
3        1
Name: count, dtype: int64


In [16]:

# Identify bridges with no recorded non-Good condition.
non_good_ids = set(first_non_good["bridge_id"])

remaining_good_ids = good_2016[
    ~good_2016.isin(non_good_ids)
]

# Find the last year each bridge was observed as Good.
last_good_year = (
    validation_history[
        validation_history["bridge_id"].isin(remaining_good_ids)
    ]
    .groupby("bridge_id")["observation_year"]
    .max()
)

print("Bridges with no non-Good observation:",
      f"{len(remaining_good_ids):,}")

print("\nLast observed Good year:")
print(last_good_year.value_counts().sort_index())

print("\nBridges never observed after 2016:",
      f"{len(remaining_good_ids) - len(last_good_year):,}")

print("\nBridges observed Good through 2020:",
      f"{(last_good_year == 2020).sum():,}")


Bridges with no non-Good observation: 237,343

Last observed Good year:
observation_year
2017       466
2018       523
2019       396
2020    234885
Name: count, dtype: int64

Bridges never observed after 2016: 1,073

Bridges observed Good through 2020: 234,885


In [17]:

# Check whether remaining-Good bridges have uninterrupted observations.
good_history = validation_history[
    validation_history["bridge_id"].isin(remaining_good_ids)
]

observation_counts = (
    good_history.groupby("bridge_id")["observation_year"].nunique()
)

# Every year from 2017 through the last observed year is expected.
expected_counts = last_good_year - 2016

missing_years_good = expected_counts - observation_counts

print("Bridges with complete follow-up:",
      f"{(missing_years_good == 0).sum():,}")

print("Bridges with observation gaps:",
      f"{(missing_years_good > 0).sum():,}")

print("\nMissing-year distribution:")
print(missing_years_good.value_counts().sort_index())

print("\nBridges with no follow-up after 2016:",
      f"{len(remaining_good_ids) - len(last_good_year):,}")


Bridges with complete follow-up: 236,164
Bridges with observation gaps: 106

Missing-year distribution:
observation_year
0    236164
1        78
2        20
3         8
Name: count, dtype: int64

Bridges with no follow-up after 2016: 1,073


In [18]:

# Construct validation outcomes from consecutive annual observations.
history_by_year = validation_history.pivot(
    index="bridge_id",
    columns="observation_year",
    values="condition"
).reindex(index=good_2016, columns=range(2017, 2021))

# Every bridge was confirmed Good at the 2016 cutoff.
validation_outcomes = pd.DataFrame({
    "bridge_id": good_2016.to_numpy(),
    "event": 0,
    "test_duration": 0,
    "stop_reason": "censored"
}).set_index("bridge_id")

active = pd.Series(True, index=validation_outcomes.index)

for year in range(2017, 2021):
    condition = history_by_year[year]
    is_deterioration = active & condition.isin(["Fair", "Poor"])
    is_good = active & condition.eq("Good")

    validation_outcomes.loc[is_deterioration, "event"] = 1
    validation_outcomes.loc[is_deterioration, "test_duration"] = year - 2016
    validation_outcomes.loc[is_deterioration, "stop_reason"] = "deterioration"

    validation_outcomes.loc[is_good, "test_duration"] = year - 2016

    # Stop following bridges after deterioration, a gap, or Unknown observation.
    active = is_good

validation_outcomes = validation_outcomes.reset_index()

print("Validation bridges:", f"{len(validation_outcomes):,}")
print("\nStop reasons:")
print(validation_outcomes["stop_reason"].value_counts())
print("\nFollow-up durations:")
print(validation_outcomes["test_duration"].value_counts().sort_index())
print("\nDeterioration events:", validation_outcomes["event"].sum())


Validation bridges: 291,414

Stop reasons:
stop_reason
censored         237368
deterioration     54046
Name: count, dtype: int64

Follow-up durations:
test_duration
0      1141
1     15178
2     15179
3     12990
4    246926
Name: count, dtype: int64

Deterioration events: 54046


In [19]:

# Check the historical validation outcomes for inconsistencies.
print("Duplicate bridge IDs:",
      validation_outcomes["bridge_id"].duplicated().sum())

print("Missing values:")
print(
    validation_outcomes[
        ["bridge_id", "event", "test_duration", "stop_reason"]
    ].isna().sum()
)

print("\nInvalid event indicators:",
      (~validation_outcomes["event"].isin([0, 1])).sum())

print("Invalid follow-up durations:",
      (~validation_outcomes["test_duration"].between(0, 4)).sum())

print("Events with zero follow-up:",
      (
          (validation_outcomes["event"] == 1)
          & (validation_outcomes["test_duration"] == 0)
      ).sum())

print("Event and stop-reason mismatches:",
      (
          (validation_outcomes["event"] == 1)
          != (validation_outcomes["stop_reason"] == "deterioration")
      ).sum())

print("\nEvents by follow-up duration:")
print(
    validation_outcomes.groupby("test_duration")["event"]
    .agg(["count", "sum"])
)


Duplicate bridge IDs: 0
Missing values:
bridge_id        0
event            0
test_duration    0
stop_reason      0
dtype: int64

Invalid event indicators: 0
Invalid follow-up durations: 0
Events with zero follow-up: 0
Event and stop-reason mismatches: 0

Events by follow-up duration:
                count    sum
test_duration               
0                1141      0
1               15178  14668
2               15179  14640
3               12990  12596
4              246926  12142


In [20]:

# Identify survival records that began before the historical cutoff.
cutoff_year = 2016

historical_candidates = train_improved.loc[
    train_improved["start_year"] < cutoff_year
].copy()

# Separate events observed by 2016 from later outcomes.
historical_candidates["historical_event"] = (
    (historical_candidates["event"] == 1)
    & (historical_candidates["end_year"] <= cutoff_year)
).astype(int)

# Restrict follow-up to information available by 2016.
historical_candidates["historical_end_year"] = (
    historical_candidates["end_year"].clip(upper=cutoff_year)
)

historical_candidates["historical_duration"] = (
    historical_candidates["historical_end_year"]
    - historical_candidates["start_year"]
)

print("Historical training candidates:",
      f"{len(historical_candidates):,}")

print("Events observed by 2016:",
      f"{historical_candidates['historical_event'].sum():,}")

print("Censored observations:",
      f"{(historical_candidates['historical_event'] == 0).sum():,}")

print("\nHistorical follow-up durations:")
print(
    historical_candidates["historical_duration"]
    .describe().round(2)
)

print("\nInvalid durations:",
      (historical_candidates["historical_duration"] <= 0).sum())


Historical training candidates: 475,720
Events observed by 2016: 207,007
Censored observations: 268,713

Historical follow-up durations:
count    475720.00
mean         10.65
std           7.78
min           1.00
25%           4.00
50%           9.00
75%          17.00
max          24.00
Name: historical_duration, dtype: float64

Invalid durations: 0


In [21]:

# Inspect the original stopping reasons for historical training records.
print("Original stopping reasons:")
print(
    historical_candidates["stop_reason"]
    .value_counts(dropna=False)
)

print("\nStopping reasons by event status:")
print(
    pd.crosstab(
        historical_candidates["stop_reason"],
        historical_candidates["event"],
        dropna=False
    )
)

print("\nRecords extending beyond the 2016 cutoff:")
future_records = historical_candidates[
    historical_candidates["end_year"] > cutoff_year
]

print("Number of records:", f"{len(future_records):,}")

print("\nStopping reasons for those records:")
print(
    future_records["stop_reason"]
    .value_counts(dropna=False)
)


Original stopping reasons:
stop_reason
Deterioration         236298
Training cutoff       147463
End of observation     75602
Structure change       10089
Observation gap         3921
Unknown condition       2347
Name: count, dtype: int64

Stopping reasons by event status:
event                    0       1
stop_reason                       
Deterioration            0  236298
End of observation   75602       0
Observation gap       3921       0
Structure change     10089       0
Training cutoff     147463       0
Unknown condition     2347       0

Records extending beyond the 2016 cutoff:
Number of records: 178,324

Stopping reasons for those records:
stop_reason
Training cutoff       147463
Deterioration          29291
End of observation       838
Structure change         654
Observation gap           78
Name: count, dtype: int64


In [22]:

# Verify the historical training outcomes after the 2016 cutoff.
print("Events after historical cutoff:",
      (
          (historical_candidates["historical_event"] == 1)
          & (historical_candidates["historical_end_year"] > cutoff_year)
      ).sum())

print("Events inconsistent with original stopping reason:",
      (
          (historical_candidates["historical_event"] == 1)
          & (historical_candidates["stop_reason"] != "Deterioration")
      ).sum())

print("Invalid historical durations:",
      (historical_candidates["historical_duration"] <= 0).sum())

print("Duplicate bridge IDs:",
      historical_candidates["bridge_id"].duplicated().sum())

print("\nHistorical outcomes:")
print(
    historical_candidates["historical_event"]
    .value_counts()
)

print("\nRecords truncated at 2016:",
      (historical_candidates["end_year"] > cutoff_year).sum())


Events after historical cutoff: 0
Events inconsistent with original stopping reason: 0
Invalid historical durations: 0
Duplicate bridge IDs: 0

Historical outcomes:
historical_event
0    268713
1    207007
Name: count, dtype: int64

Records truncated at 2016: 178324


In [23]:

# Examine overlap between historical training and validation bridges.
training_ids = set(historical_candidates["bridge_id"])
validation_ids = set(validation_outcomes["bridge_id"])

overlap_ids = training_ids & validation_ids

print("Historical training bridges:",
      f"{len(training_ids):,}")

print("Historical validation bridges:",
      f"{len(validation_ids):,}")

print("Bridges appearing in both:",
      f"{len(overlap_ids):,}")

# Identify overlapping bridges with a recorded pre-2017 deterioration.
previously_deteriorated = historical_candidates[
    (historical_candidates["bridge_id"].isin(overlap_ids))
    & (historical_candidates["historical_event"] == 1)
]

print("\nOverlapping bridges with deterioration by 2016:",
      f"{len(previously_deteriorated):,}")

print("\nTheir original stopping reasons:")
print(previously_deteriorated["stop_reason"].value_counts())


Historical training bridges: 475,720
Historical validation bridges: 291,414
Bridges appearing in both: 219,042

Overlapping bridges with deterioration by 2016: 32,874

Their original stopping reasons:
stop_reason
Deterioration    32874
Name: count, dtype: int64


In [24]:

# Identify bridges with a recorded deterioration before validation.
previous_event_ids = set(
    historical_candidates.loc[
        historical_candidates["historical_event"] == 1,
        "bridge_id"
    ]
)

# Retain bridges without a recorded pre-2017 deterioration.
validation_first_event = validation_outcomes.loc[
    ~validation_outcomes["bridge_id"].isin(previous_event_ids)
].copy()

# Exclude bridges without any follow-up after 2016.
validation_first_event = validation_first_event.loc[
    validation_first_event["test_duration"] > 0
].copy()

print("Original validation bridges:",
      f"{len(validation_outcomes):,}")

print("Bridges with previous deterioration:",
      f"{len(validation_outcomes) - len(validation_outcomes.loc[~validation_outcomes['bridge_id'].isin(previous_event_ids)]):,}")

print("Final eligible validation bridges:",
      f"{len(validation_first_event):,}")

print("\nValidation deterioration events:",
      f"{validation_first_event['event'].sum():,}")

print("Validation censored bridges:",
      f"{(validation_first_event['event'] == 0).sum():,}")

print("\nEvent rate:",
      round(validation_first_event["event"].mean() * 100, 2), "%")


Original validation bridges: 291,414
Bridges with previous deterioration: 32,874
Final eligible validation bridges: 257,549

Validation deterioration events: 44,504
Validation censored bridges: 213,045

Event rate: 17.28 %


In [25]:

# Calculate time already spent in Good condition at the 2016 cutoff.
historical_entry = historical_candidates[
    ["bridge_id", "start_year"]
].rename(columns={"start_year": "historical_start_year"})

validation_with_entry = validation_first_event.merge(
    historical_entry,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

validation_with_entry["entry_duration"] = (
    2016 - validation_with_entry["historical_start_year"]
)

print("Validation bridges:", f"{len(validation_with_entry):,}")

print("\nBridges with known historical start year:",
      f"{validation_with_entry['historical_start_year'].notna().sum():,}")

print("Bridges without historical start year:",
      f"{validation_with_entry['historical_start_year'].isna().sum():,}")

print("\nHistorical entry durations:")
print(validation_with_entry["entry_duration"].describe().round(2))

print("\nInvalid entry durations:",
      (validation_with_entry["entry_duration"] < 0).sum())


Validation bridges: 257,549

Bridges with known historical start year: 185,604
Bridges without historical start year: 71,945

Historical entry durations:
count    185604.00
mean         14.32
std           8.64
min           1.00
25%           5.00
50%          14.00
75%          24.00
max          24.00
Name: entry_duration, dtype: float64

Invalid entry durations: 0


In [26]:

# Check historical coverage for validation bridges.
historical_lookup = historical_candidates[
    ["bridge_id", "start_year", "historical_end_year",
     "historical_event", "stop_reason"]
]

entry_check = validation_first_event[
    ["bridge_id"]
].merge(
    historical_lookup,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

matched = entry_check["start_year"].notna()

print("Validation bridges with historical records:",
      f"{matched.sum():,}")

print("Validation bridges without historical records:",
      f"{(~matched).sum():,}")

print("\nMatched records ending before 2016:",
      f"{(matched & (entry_check['historical_end_year'] < 2016)).sum():,}")

print("Matched records reaching 2016:",
      f"{(matched & (entry_check['historical_end_year'] == 2016)).sum():,}")

print("\nStopping reasons for records ending before 2016:")
print(
    entry_check.loc[
        matched & (entry_check["historical_end_year"] < 2016),
        "stop_reason"
    ].value_counts()
)


Validation bridges with historical records: 185,604
Validation bridges without historical records: 71,945

Matched records ending before 2016: 7,053
Matched records reaching 2016: 178,551

Stopping reasons for records ending before 2016:
stop_reason
Structure change     4948
Observation gap      1296
Unknown condition     809
Name: count, dtype: int64


In [27]:

# Examine the previous year's condition for validation bridges.
snapshot_2015 = pd.read_parquet(
    PROJECT / "data" / "processed" / "nbi_2015_clean.parquet",
    columns=["bridge_id", "condition"]
)

previous_conditions = validation_first_event[
    ["bridge_id"]
].merge(
    snapshot_2015.rename(
        columns={"condition": "condition_2015"}
    ),
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

print("2015 conditions for eligible validation bridges:")
print(
    previous_conditions["condition_2015"]
    .value_counts(dropna=False)
)

print("\nBridges Good in both 2015 and 2016:",
      f"{(previous_conditions['condition_2015'] == 'Good').sum():,}")

print("Bridges not recorded in 2015:",
      f"{previous_conditions['condition_2015'].isna().sum():,}")


2015 conditions for eligible validation bridges:
condition_2015
Good       244136
NaN          9146
Fair         3183
Poor         1075
Unknown         9
Name: count, dtype: int64

Bridges Good in both 2015 and 2016: 244,136
Bridges not recorded in 2015: 9,146


In [28]:

# Find the start of each bridge's current continuous Good episode.
eligible_ids = validation_first_event["bridge_id"]

episode_start = pd.Series(
    2016,
    index=eligible_ids.to_numpy(),
    name="episode_start_year"
)

# Only bridges still continuously Good can extend into earlier years.
active_ids = pd.Index(eligible_ids)

for year in range(2015, 1991, -1):
    snapshot = pd.read_parquet(
        PROJECT / "data" / "processed" / f"nbi_{year}_clean.parquet",
        columns=["bridge_id", "condition"]
    )

    conditions = snapshot.set_index("bridge_id")["condition"]
    good_ids = active_ids.intersection(
        conditions.index[conditions.eq("Good")]
    )

    episode_start.loc[good_ids] = year
    active_ids = good_ids

    print(f"{year}: {len(active_ids):,} bridges still continuously Good")

validation_episode_start = episode_start.rename_axis(
    "bridge_id"
).reset_index()

print("\nEpisode start year distribution:")
print(
    validation_episode_start["episode_start_year"]
    .value_counts().sort_index()
)

print("\nTotal bridges:",
      f"{len(validation_episode_start):,}")


2015: 244,136 bridges still continuously Good
2014: 234,683 bridges still continuously Good
2013: 213,618 bridges still continuously Good
2012: 201,889 bridges still continuously Good
2011: 193,696 bridges still continuously Good
2010: 178,687 bridges still continuously Good
2009: 171,874 bridges still continuously Good
2008: 165,394 bridges still continuously Good
2007: 159,286 bridges still continuously Good
2006: 152,677 bridges still continuously Good
2005: 142,815 bridges still continuously Good
2004: 135,890 bridges still continuously Good
2003: 129,708 bridges still continuously Good
2002: 123,885 bridges still continuously Good
2001: 112,785 bridges still continuously Good
2000: 106,507 bridges still continuously Good
1999: 98,282 bridges still continuously Good
1998: 92,914 bridges still continuously Good
1997: 88,306 bridges still continuously Good
1996: 79,891 bridges still continuously Good
1995: 75,276 bridges still continuously Good
1994: 70,225 bridges still continuously

In [29]:

# Attach current Good-episode start years to historical validation outcomes.
validation_episode = validation_first_event.merge(
    validation_episode_start,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

validation_episode["entry_duration"] = (
    2016 - validation_episode["episode_start_year"]
)

print("Validation bridges:", f"{len(validation_episode):,}")

print("\nMissing entry durations:",
      validation_episode["entry_duration"].isna().sum())

print("Invalid entry durations:",
      (~validation_episode["entry_duration"].between(0, 24)).sum())

print("\nEntry duration summary:")
print(validation_episode["entry_duration"].describe().round(2))

print("\nEntry duration distribution:")
print(
    validation_episode["entry_duration"]
    .value_counts().sort_index()
)


Validation bridges: 257,549

Missing entry durations: 0
Invalid entry durations: 0

Entry duration summary:
count    257549.00
mean         12.80
std           8.64
min           0.00
25%           5.00
50%          13.00
75%          23.00
max          24.00
Name: entry_duration, dtype: float64

Entry duration distribution:
entry_duration
0     13413
1      9453
2     21065
3     11729
4      8193
5     15009
6      6813
7      6480
8      6108
9      6609
10     9862
11     6925
12     6182
13     5823
14    11100
15     6278
16     8225
17     5368
18     4608
19     8415
20     4615
21     5051
22     4229
23     8795
24    57201
Name: count, dtype: int64


In [30]:

# Compare current Good-episode starts with original survival starts.
original_histories = historical_candidates[
    ["bridge_id", "start_year", "historical_end_year", "historical_event"]
].rename(columns={"start_year": "original_start_year"})

episode_consistency = validation_episode.merge(
    original_histories,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

# Require an original history reaching 2016 with the same start year.
consistent = (
    episode_consistency["original_start_year"].notna()
    & (episode_consistency["original_start_year"]
       == episode_consistency["episode_start_year"])
    & (episode_consistency["historical_end_year"] == 2016)
    & (episode_consistency["historical_event"] == 0)
)

print("Total validation bridges:",
      f"{len(episode_consistency):,}")

print("Bridges consistent with original survival histories:",
      f"{consistent.sum():,}")

print("Bridges excluded by consistency check:",
      f"{(~consistent).sum():,}")

print("\nDeterioration events among consistent bridges:",
      f"{episode_consistency.loc[consistent, 'event'].sum():,}")

print("\nEvent rate among consistent bridges:",
      round(100 * episode_consistency.loc[consistent, "event"].mean(), 2),
      "%")


Total validation bridges: 257,549
Bridges consistent with original survival histories: 178,551
Bridges excluded by consistency check: 78,998

Deterioration events among consistent bridges: 29,369

Event rate among consistent bridges: 16.45 %


In [31]:

# Retain bridges whose histories match the original survival definition.
historical_validation = (
    episode_consistency.loc[
        consistent,
        [
            "bridge_id",
            "event",
            "test_duration",
            "entry_duration",
            "episode_start_year"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Calculate total survival time at the end of validation follow-up.
historical_validation["total_duration"] = (
    historical_validation["entry_duration"]
    + historical_validation["test_duration"]
)

print("Historical validation bridges:",
      f"{len(historical_validation):,}")

print("Deterioration events:",
      f"{historical_validation['event'].sum():,}")

print("\nEntry duration summary:")
print(historical_validation["entry_duration"].describe())

print("\nTotal duration summary:")
print(historical_validation["total_duration"].describe())

print("\nMissing values:")
print(historical_validation.isna().sum())


Historical validation bridges: 178,551
Deterioration events: 29,369

Entry duration summary:
count    178551.000000
mean         14.120711
std           8.661973
min           1.000000
25%           5.000000
50%          14.000000
75%          24.000000
max          24.000000
Name: entry_duration, dtype: float64

Total duration summary:
count    178551.000000
mean         17.860695
std           8.650713
min           2.000000
25%           9.000000
50%          18.000000
75%          28.000000
max          28.000000
Name: total_duration, dtype: float64

Missing values:
bridge_id             0
event                 0
test_duration         0
entry_duration        0
episode_start_year    0
total_duration        0
dtype: int64


In [32]:

# Retrieve baseline predictors from the historical survival records.
historical_predictors = historical_candidates[
    ["bridge_id"] + all_predictors
].copy()

# Attach predictors to the historical validation outcomes.
historical_validation = historical_validation.merge(
    historical_predictors,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

print("Historical validation bridges:",
      f"{len(historical_validation):,}")

print("\nPredictor columns:")
print(all_predictors)

print("\nMissing values by predictor:")
print(historical_validation[all_predictors].isna().sum())

print("\nValidation events:",
      f"{historical_validation['event'].sum():,}")

print("\nDuplicate bridge IDs:",
      historical_validation["bridge_id"].duplicated().sum())


Historical validation bridges: 178,551

Predictor columns:
['bridge_age', 'ADT_029', 'STRUCTURE_LEN_MT_049', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B', 'was_reconstructed', 'years_since_reconstruction']

Missing values by predictor:
bridge_age                    326
ADT_029                       501
STRUCTURE_LEN_MT_049           32
STRUCTURE_KIND_043A            48
STRUCTURE_TYPE_043B           149
was_reconstructed               0
years_since_reconstruction      0
dtype: int64

Validation events: 29,369

Duplicate bridge IDs: 0


In [33]:

# Prepare survival training records using outcomes available by 2016.
historical_train = historical_candidates.copy()

historical_train["event"] = historical_train["historical_event"]
historical_train["duration"] = historical_train["historical_duration"]

# Retain identifiers, survival outcomes and model predictors.
historical_train = historical_train[
    ["bridge_id", "duration", "event"] + all_predictors
].copy()

print("Historical training bridges:", f"{len(historical_train):,}")
print("Historical training events:", f"{historical_train['event'].sum():,}")

print("\nTraining durations:")
print(historical_train["duration"].describe().round(2))

print("\nMissing predictor values:")
print(historical_train[all_predictors].isna().sum())

print("\nDuplicate bridge IDs:",
      historical_train["bridge_id"].duplicated().sum())


Historical training bridges: 475,720
Historical training events: 207,007

Training durations:
count    475720.00
mean         10.65
std           7.78
min           1.00
25%           4.00
50%           9.00
75%          17.00
max          24.00
Name: duration, dtype: float64

Missing predictor values:
bridge_age                    1413
ADT_029                       1978
STRUCTURE_LEN_MT_049           230
STRUCTURE_KIND_043A            348
STRUCTURE_TYPE_043B            389
was_reconstructed                0
years_since_reconstruction       0
dtype: int64

Duplicate bridge IDs: 0


In [34]:

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define the original and improved predictors.
original_predictors = [
    "bridge_age",
    "ADT_029",
    "STRUCTURE_LEN_MT_049",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

improved_predictors = original_predictors + [
    "was_reconstructed",
    "years_since_reconstruction"
]

categorical_predictors = [
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

def prepare_features(predictors):
    numerical = [
        col for col in predictors
        if col not in categorical_predictors
    ]

    # Convert pandas missing values to NumPy NaN.
    train_features = historical_train[predictors].copy()
    val_features = historical_validation[predictors].copy()

    for df in [train_features, val_features]:
        for col in numerical:
            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            ).astype(float)

        for col in categorical_predictors:
            df[col] = (
                df[col]
                .astype("string")
                .fillna("__missing__")
                .astype(str)
            )

    preprocessor = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numerical),
        ("categorical", Pipeline([
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]), categorical_predictors)
    ])

    X_train = preprocessor.fit_transform(train_features)
    X_validation = preprocessor.transform(val_features)

    return X_train, X_validation, preprocessor

X_train_original, X_val_original, prep_original = (
    prepare_features(original_predictors)
)

X_train_improved, X_val_improved, prep_improved = (
    prepare_features(improved_predictors)
)

print("Original model:")
print("Training shape:", X_train_original.shape)
print("Validation shape:", X_val_original.shape)

print("\nImproved model:")
print("Training shape:", X_train_improved.shape)
print("Validation shape:", X_val_improved.shape)


Original model:
Training shape: (475720, 38)
Validation shape: (178551, 38)

Improved model:
Training shape: (475720, 40)
Validation shape: (178551, 40)


In [35]:

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Scale features using historical training data only.
scaler_original = StandardScaler()
scaler_improved = StandardScaler()

X_train_original_scaled = scaler_original.fit_transform(X_train_original)
X_train_improved_scaled = scaler_improved.fit_transform(X_train_improved)

# Create survival training dataframes.
cox_train_original = pd.DataFrame(
    X_train_original_scaled,
    columns=[f"feature_{i}" for i in range(X_train_original.shape[1])]
)

cox_train_improved = pd.DataFrame(
    X_train_improved_scaled,
    columns=[f"feature_{i}" for i in range(X_train_improved.shape[1])]
)

for df in [cox_train_original, cox_train_improved]:
    df["duration"] = historical_train["duration"].to_numpy()
    df["event"] = historical_train["event"].to_numpy()

# Fit both models with the same regularisation.
cox_original = CoxPHFitter(penalizer=0.01)
cox_improved = CoxPHFitter(penalizer=0.01)

print("Training original Cox model...")
cox_original.fit(
    cox_train_original,
    duration_col="duration",
    event_col="event"
)

print("Training improved Cox model...")
cox_improved.fit(
    cox_train_improved,
    duration_col="duration",
    event_col="event"
)

print("\nBoth Cox models trained successfully.")
print("Original training C-index:", round(cox_original.concordance_index_, 4))
print("Improved training C-index:", round(cox_improved.concordance_index_, 4))


Training original Cox model...
Training improved Cox model...

Both Cox models trained successfully.
Original training C-index: 0.6776
Improved training C-index: 0.6789


In [36]:
import numpy as np
from lifelines.utils import concordance_index

# Scale validation features using the fitted training scalers.
X_val_original_scaled = scaler_original.transform(X_val_original)
X_val_improved_scaled = scaler_improved.transform(X_val_improved)

def evaluate_historical_cox(model, X_scaled):
    feature_names = list(model.params_.index)

    validation_features = pd.DataFrame(
        X_scaled,
        columns=feature_names
    )

    # Higher relative hazard = earlier predicted deterioration.
    relative_hazard = model.predict_partial_hazard(
        validation_features
    ).to_numpy().ravel()

    # Rank bridges by relative hazard (negated for lifelines).
    return concordance_index(
        historical_validation["test_duration"],
        -relative_hazard,
        event_observed=historical_validation["event"]
    )

original_val_cindex = evaluate_historical_cox(
    cox_original, X_val_original_scaled
)

improved_val_cindex = evaluate_historical_cox(
    cox_improved, X_val_improved_scaled
)

print("Historical validation C-index (hazard ranking)")
print("Original Cox:", round(original_val_cindex, 4))
print("Improved Cox:", round(improved_val_cindex, 4))
print("Improvement:", round(improved_val_cindex - original_val_cindex, 4))


Historical validation C-index (hazard ranking)
Original Cox: 0.6068
Improved Cox: 0.6085
Improvement: 0.0017


In [37]:

from pathlib import Path
import pandas as pd

results_dir = RESULTS
results_dir.mkdir(parents=True, exist_ok=True)

improvement_results = pd.DataFrame({
    "model": ["Original Cox PH", "Improved Cox PH"],
    "predictor_count": [5, 7],
    "training_cindex": [cox_original.concordance_index_, cox_improved.concordance_index_],
    "historical_validation_cindex": [
        original_val_cindex,
        improved_val_cindex
    ],
    "validation_bridges": [len(historical_validation)] * 2,
    "validation_events": [int(historical_validation["event"].sum())] * 2
})

improvement_results.to_csv(
    results_dir / "feature_improvement_comparison.csv",
    index=False
)

print(improvement_results.round(4).to_string(index=False))
print("\nResults saved successfully.")


          model  predictor_count  training_cindex  historical_validation_cindex  validation_bridges  validation_events
Original Cox PH                5           0.6776                        0.6068              178551              29369
Improved Cox PH                7           0.6789                        0.6085              178551              29369

Results saved successfully.
